# Notebook to test simulator

notebook must be moved out of the test folder, otherwise the R-simulator will not find the necessary files.

In [2]:
import numpy as np
import pandas as pd
from rpy2.robjects import r, conversion, pandas2ri
from helper_functions import normalize_household_data, dict_to_named_list#
import matplotlib.pyplot as plt

In [3]:
pandas2ri.activate()
r.source('Simulator/Simulator.R')
model_r = r['simulate_and_reformat']

R[write to console]: 
Attaching package: ‘dplyr’


R[write to console]: The following objects are masked from ‘package:stats’:

    filter, lag


R[write to console]: The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


R[write to console]: 
Attaching package: ‘actuar’


R[write to console]: The following objects are masked from ‘package:stats’:

    sd, var


R[write to console]: The following object is masked from ‘package:grDevices’:

    cm




In [4]:
PARAM_NAMES = [
    'alpha',  # alpha is not estimated
    'beta', 'delta',
    'mu_inf_SC', 'mu_inf_SI',
    'mu_inf_AI', 'mu_inf_AC', 'mu_inf_AA',
    'mu_susc_C', 'mu_susc_I',
    'mu_protect_acq',  # fixed
    'mu_protect_transm',  # fixed
    'mu_low_conf', 'mu_high_conf'
]

PROCEDURES = ['pedcov', 'random', 'original_pedcov', 'original_random'] #, 'adult', 'sampling1', 'samplingIG']

In [5]:
def prior(batch_size: int) -> np.ndarray:
    param_batch = np.random.normal(0, 1, (batch_size, len(PARAM_NAMES)))
    # alpha
    param_batch[:, 0] = np.random.uniform(0, 0.1, batch_size)
    return param_batch

test = prior(1).flatten()

In [6]:
def simulator(log_params: np.ndarray,
              selection_procedure: str,
              variant: str,
              minimal_length: int = 9,
              fixed_parameters_dict: dict = None) -> np.ndarray:
    """
    Simulate data with given parameters and reformat it to a numpy array.
    :param log_params: parameters for the simulation
    :param selection_procedure: selection procedure for the simulation (pedcov or random)
    :param variant: variant of the simulation (alpha or omicron)
    :param minimal_length: minimal length of the data
    :param fixed_parameters_dict: dictionary with fixed parameters for each variant
    :return: simulated data as numpy array
    """
    if variant not in ['alpha', 'omicron']:
        raise ValueError(f"Variant '{variant}' not supported. Must be 'alpha' or 'omicron'.")
    if selection_procedure not in PROCEDURES:
        raise ValueError(f"Selection procedure '{selection_procedure}' not supported. "
                         f"Must be one of {PROCEDURES}.")

    if fixed_parameters_dict is None:
        fixed_parameters = []
    else:
        fixed_parameters = fixed_parameters_dict[variant]

    # transform parameters to correct scale
    un_scaled_params = np.copy(log_params)  # copy to avoid changing input
    # create dict from param_names, params might have different length
    par_dict = {}
    p_i = 0
    for name in PARAM_NAMES:
        if name in fixed_parameters:
            par_dict.update({name: fixed_parameters[name]})
        # all parameters besides alpha and delta are log-transformed
        elif name == 'delta' or name == 'alpha':
            par_dict.update({name: un_scaled_params[p_i]})
            p_i += 1
        else:
            par_dict.update({name: np.exp(un_scaled_params[p_i])})
            p_i += 1
    # update dict with fixed hyperparameters, make sure these are strings
    par_dict.update({'variant': str(variant), 'selection_procedure': str(selection_procedure)})

    # simulate data
    sim_data_r = model_r(dict_to_named_list(par_dict))
    # convert to pandas dataframe
    sim_data_full = conversion.rpy2py(sim_data_r)
    # normalize data and return as numpy array
    #sim_data_norm = normalize_household_data(sim_data_full, minimal_length=minimal_length)
    return sim_data_full

In [7]:
sim_data_full = simulator(test, selection_procedure='random', variant='alpha')
sim_data_full.shape

(538, 15)

In [8]:
sim_data_full['conf'].value_counts()

conf
0    399
2    105
1     34
Name: count, dtype: int64

In [9]:
sim_data = normalize_household_data(sim_data_full, minimal_length=9)
sim_data[0]

     id_patient        id_hh id_hh_origin  hh_size  date_sympt  infect_status  \
24         6378  119-a-12-p1          119        6   51.687408            2.0   
48        17853   48-a-34-p1           48        5   51.824253            2.0   
121       23646   27-a-45-p1           27        3   43.631229            2.0   
122       23647   27-a-45-p1           27        3   90.631229            2.0   
182       19483   55-a-37-p1           55        5   36.000000            2.0   
238        2088   116-a-4-p1          116        4   48.000000            2.0   
249       22134   48-a-42-p1           48        5   42.000000            2.0   
289         419   100-a-1-p1          100        4   45.988581            2.0   
383        9119    7-a-18-p1            7        4   40.000000            2.0   
386        9122    7-a-18-p1            7        4   34.000000            2.0   
441       22375  106-a-42-p1          106        6   48.000000            2.0   
464       26645  103-a-50-p1

ValueError: Error processing household data: npi_stop is before date_sympt

In [ ]:
# check inclusion cases
# for each household get first infected individual and extract age, infection status
count_inclusion_cases_age = {0: 0, 1: 0, 2: 0}
count_symptomatic_cases = {0: 0, 1: 0, 2: 0}

for hh_id in sim_data['id_hh'].unique():
    hh_data = sim_data[sim_data['id_hh'] == hh_id]
    # check if there are multiple infection dates
    first_infected = hh_data.loc[hh_data['date_sympt'] == np.min(hh_data['date_sympt'])]
    min_age = int(np.min(first_infected['age']))
    min_status = int(np.min(first_infected['infect_status']))
    count_inclusion_cases_age[min_age] += 1
    count_symptomatic_cases[min_status] += 1
    #print(hh_data)
    
# plot cases
plt.bar(count_inclusion_cases_age.keys(), count_inclusion_cases_age.values())
plt.xticks([0, 1, 2], labels=['Infant', 'Child', 'Adult'])
plt.show()

plt.bar(count_symptomatic_cases.keys(), count_symptomatic_cases.values())
plt.xticks([0, 1, 2], labels=['X', 'symptomatic', 'asymptomatic'])
plt.show()

In [ ]:
sim_data_full['hh_size'].max()

In [ ]:
sim_data = normalize_household_data(sim_data_full, minimal_length=9)
sim_data[0]

In [ ]:
# check inclusion cases
# for each household get first infected individual and extract age, infection status
count_inclusion_cases_age = {0: 0, 1: 0, 2: 0, 3: 0}
count_symptomatic_cases = {0: 0, 1: 0, 2: 0}

for hh_id in sim_data_full['id_hh'].unique():
    hh_data = sim_data_full[sim_data_full['id_hh'] == hh_id]
    # check if there are multiple infection dates
    first_infected = hh_data.loc[hh_data['date_sympt'] == np.min(hh_data['date_sympt'])]
    min_age = int(np.min(first_infected['age_exact']))
    if min_age < 6:
        min_age_group = 0
    elif min_age <= 11:
        min_age_group = 1
    elif min_age <= 18:  # different to the age_group in the data
        min_age_group = 2
    else:
        min_age_group = 3
    min_status = int(np.min(first_infected['infect_status']))
    count_inclusion_cases_age[min_age_group] += 1
    count_symptomatic_cases[min_status] += 1
    #print(hh_data)
    
# plot cases
plt.bar(count_inclusion_cases_age.keys(), count_inclusion_cases_age.values())
plt.xticks([0, 1, 2, 3], labels=['Infant', 'Child', 'Adolescent', 'Adult'])
plt.show()

plt.bar(count_symptomatic_cases.keys(), count_symptomatic_cases.values())
plt.xticks([0, 1, 2], labels=['X', 'symptomatic', 'asymptomatic'])
plt.show()

In [ ]:
sim_data_full[sim_data_full['id_hh_origin'] == "10"]

In [ ]:
sim_data_full['id_hh_origin'].unique()

In [ ]:
# extract the original household ids
sim_data_full['id_hh_origin'].nunique()

In [ ]:
# columns: date_sympt_norm, infect_status_norm, age_norm, protected
# rows: empty individuals in the beginning (households are of same size)
# last row: end_followup_norm with 1

In [ ]:
# load a presimulation using pickle
import pickle
with open('presimulations/presim_file_1.pkl', 'rb') as f:
    presim = pickle.load(f)
    
with open('presimulations/presim_file_2.pkl', 'rb') as f:
    presim_2 = pickle.load(f)

In [ ]:
presim[0]['prior_draws']#-presim_2[5]['prior_draws']

In [ ]:
len(presim)

In [ ]:
presim_2[0]['prior_draws']#[:, :, :, 0]*1000

In [ ]:
with open('valid_data.pickle', 'rb') as f:
    valid_data = pickle.load(f)

In [ ]:
valid_data['prior_draws']